In [12]:
import pandas as pd
import numpy as np
import re
df=pd.read_csv('messysupermarket_sales(1).csv')

In [13]:
df.isnull().sum()

Sales_ID           0
Date              37
Custumer_Name     63
Product           39
Category          75
Quantity          36
Unit_Price        50
Total             60
Payment_Method    58
dtype: int64

In [14]:
text_col=['Custumer_Name','Product','Category','Payment_Method']
for col in text_col:
    if col in df.columns:
        df[col]=df[col].astype('string').str.strip().str.title()

In [15]:
# filling empty columns custumer name

for col in ['Custumer_Name']:
    if col in df.columns:
        df[col]=df[col].replace(['nan',''],np.nan).fillna('name')
   

In [16]:
# filling empty columns

for col in['Category','Product','Payment_Method']:
    if col in df.columns:
        df[col]=df[col].replace(['nan',''],np.nan).fillna('unknown')

In [17]:
# replacing the text in quantity columns

df['Quantity']=df['Quantity'].astype(str).str.replace(r'[^\d.]','',regex=True)
df['Quantity']=pd.to_numeric(df['Quantity'],errors='coerce').fillna(0.0)

In [18]:
# handling the unit_price and total columns

for col in['Unit_Price','Total']:
    if col in df.columns:
        df[col]=df[col].astype(str).replace(r'[^\d.]','',regex=True)
        df[col]=pd.to_numeric(df[col],errors='coerce').fillna(0.0)

In [19]:
df.dropna(subset=['Sales_ID','Custumer_Name'])

,Sales_ID,Date,Custumer_Name,Product,Category,Quantity,Unit_Price,Total,Payment_Method
0,S1,4th February 2026,Miriam,Cabbage,Vegetable,5.0,3.00,15.00,Cash
1,S2,4th February 2026,Miriam,Champagne,Beverages,3.0,0.00,45.00,Cash
2,S3,4/2/2026,Miriam,unknown,Fruits,1.0,5.00,150.00,Cash
3,S4,4/2/2026,name,Grapes,Fruits,3.0,4.00,12.00,Cash
4,S5,4/2/2026,Miriam,Apples,unknown,13.0,3.00,39.00,Cash
...,...,...,...,...,...,...,...,...,...
347,O40,28-Feb-26,name,Potatoes,unknown,2.0,8.00,16.00,Bank Transfer
348,O41,28-Feb-26,Tyler,unknown,Household,4.0,3.25,0.00,Credit Card
349,O42,28-Feb-26,Paige Briggs,Sausage,Personal Care,2.0,1.50,3.00,Credit Card
350,O43,28-Feb-26,Travis Sams,Rice,unknown,1.0,0.00,0.00,Paypal


In [20]:
# handling date columns

def clean_date_string(val):

    if pd.isna(val) or str(val).strip().lower() in ['', 'nan', 'none']:
        return np.nan

    val = str(val).strip().lower()

    # Remove commas
    val = val.replace(",", "")

    # Remove ordinal suffixes
    val = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', val)

    # Correct spelling mistakes
    corrections = {
        "febuary": "february",
        "feburary": "february",
        "febrary": "february",
        "febraray": "february",
        "februaru": "february",
        "o2": "02"
    }

    for wrong, correct in corrections.items():
        val = val.replace(wrong, correct)

    # Replace standalone "fe" with "feb"
    val = re.sub(r'\bfe\b', 'feb', val)

    # Standardize separators
    val = re.sub(r"[.\-\s]", "/", val)

    if re.fullmatch(r"\d{1,2}[/-][a-z]+", val):
        val += "/2026"

    return val

df["Date"] = df["Date"].apply(clean_date_string)

df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [21]:
df.to_csv('cleaned_supermarket_sales.csv',index=False)